# 12 — Rule-backed disciplines

**The concept:** a discipline whose behaviour is a **reviewed file of rules** rather than a Python
function. A language model may have drafted the file, but what runs is the file — deterministic,
inspectable, diffable, version-controlled.

This targets the gap gradient-based MDO cannot reach: **discrete architectural choice**. Which spar
material, how many ribs, which propulsion architecture — questions with no gradient, where the
answer is a selection rather than a number.

### First, what is clingo?

The rules are written in **ASP (Answer Set Programming)**, and **clingo** is the solver that runs
them. If neither term is familiar, this is the whole of what you need:

- You state **facts** (`mass_band(heavy).`) and **rules** about what may hold together.
- A rule beginning `:-` is a **constraint** — it reads as *"never this"*. So
  `:- spar(aluminium), mass_band(heavy).` means *an aluminium spar and a heavy airframe cannot
  co-exist.*
- `1 { spar(M) : material(M) } 1.` is a **choice rule**: pick exactly one `spar` from the materials.
- clingo finds every assignment satisfying all of it. Each is an **answer set** — here, one complete
  architecture.
- If nothing satisfies the rules, the answer is **UNSAT**, which is a *proof* that no architecture
  works rather than a failure to find one.

That is enough to read every program in this notebook. It is deliberately a small language: the
point is that an engineer can review the rules without learning a framework.

Needs the extra: `pip install 'smartmdao[asp]'` — which installs clingo.

In [1]:
import pathlib, tempfile
from smartmdao import RuleDiscipline, INFEASIBLE

workspace = pathlib.Path(tempfile.mkdtemp())
program = workspace / "spar.lp"
program.write_text(
    "material(aluminium; cfrp).\n"
    "1 { spar(M) : material(M) } 1.\n"
    ":- spar(aluminium), mass_band(heavy).\n"
    ":- spar(cfrp), certification(part25).\n"
    "cost(aluminium, 1). cost(cfrp, 3).\n"
    "#minimize { C@1,M : spar(M), cost(M,C) }.\n"
    "rank(aluminium,1). rank(cfrp,2).\n"
    "#minimize { R@0,M : spar(M), rank(M,R) }.\n"
    "#show spar/1.\n"
)
print(program.read_text())

material(aluminium; cfrp).
1 { spar(M) : material(M) } 1.
:- spar(aluminium), mass_band(heavy).
:- spar(cfrp), certification(part25).
cost(aluminium, 1). cost(cfrp, 3).
#minimize { C@1,M : spar(M), cost(M,C) }.
rank(aluminium,1). rank(cfrp,2).
#minimize { R@0,M : spar(M), rank(M,R) }.
#show spar/1.



Read that file. Those constraints are the discipline: *aluminium cannot carry a heavy
airframe; CFRP is not certified to Part 25.* An engineer can agree or disagree with them, which is
the entire point — a prompt cannot be reviewed that way.

In [2]:
rules = RuleDiscipline(program, facts=["mass_band", "certification"], produces="decisions")

print("step name:", rules.name)
print("facts in: ", rules.facts)
print("produces: ", rules.produces)
print()
for band in ("light", "heavy"):
    for cert in ("part23", "part25"):
        answer = rules.solve(mass_band=band, certification=cert)
        rendered = "INFEASIBLE" if answer is INFEASIBLE else ", ".join(sorted(answer))
        print(f"  {band:6} {cert:7} -> {rendered}")

step name: rules_spar
facts in:  ('mass_band', 'certification')
produces:  decisions

  light  part23  -> spar(aluminium)
  light  part25  -> spar(aluminium)
  heavy  part23  -> spar(cfrp)
  heavy  part25  -> INFEASIBLE


## The output is a frozenset of atoms

Which is why it couples straight into a feedback loop: structural equality over a set of decisions,
with no prose in it to perturb. The existing non-numeric convergence handles it with **no solver
change at all**.

In [3]:
answer = rules.solve(mass_band="light", certification="part23")
print(type(answer).__name__, sorted(answer))

frozenset ['spar(aluminium)']


## UNSAT is the honest INFEASIBLE

Heavy *and* Part 25 rules out both materials. That is not a sentinel anyone invented — clingo
**proved** no model satisfies the constraints. It is an explicit value, never `None`, because a step
returning `None` stores nothing and reads as converged.

In [4]:
blocked = rules.solve(mass_band="heavy", certification="part25")
print("value:     ", repr(blocked))
print("is None?   ", blocked is None)
print("is falsey? ", not blocked)

value:      INFEASIBLE
is None?    False
is falsey?  True


## Why there is no answer

`explain_infeasible` returns the **minimal** set of facts that cannot hold together — remove any one
of them and the rules become satisfiable.

In [5]:
from smartmdao import Conflict

conflict = rules.explain_infeasible(mass_band="heavy", certification="part25")
print(conflict)
print()
print("type:       ", type(conflict).__name__, "| is a Conflict:", isinstance(conflict, Conflict))
print("facts:      ", conflict.facts)
print("rules alone:", conflict.rules_alone)
print("cost:       ", conflict.solves, "solves")

these facts cannot hold together: mass_band(heavy), certification(part25). Removing any one of them makes the rules satisfiable

type:        Conflict | is a Conflict: True
facts:       {'mass_band': 'heavy', 'certification': 'part25'}
rules alone: False
cost:        3 solves


If the program contradicts itself regardless of input, `rules_alone` is `True` and `facts` is
empty — meaning no input could have worked, so read the program rather than your requirements.

In [6]:
impossible = workspace / "impossible.lp"
impossible.write_text(":- not spar(x).\n:- spar(x).\nseen(B) :- mass_band(B).\n#show spar/1.\n")

broken = RuleDiscipline(impossible, facts=["mass_band"], produces="decisions")
print(broken.explain_infeasible(mass_band="light"))

the rules are unsatisfiable on their own: no facts are involved, so no input could have made this work


It is **on demand, not automatic** — one solve per fact, so computing it on every sweep of a
loop would charge for an explanation nobody read.

## A program that does not pin its own answer refuses to guess

Two equally optimal answer sets means clingo's search order decides your architecture. That is a
finding, not a detail, so it raises.

In [7]:
from smartmdao import AmbiguousProgramError

tied = workspace / "tied.lp"
tied.write_text(
    "material(aluminium; cfrp).\n"
    "1 { spar(M) : material(M) } 1.\n"
    "cost(aluminium, 2). cost(cfrp, 2).\n"
    "#minimize { C,M : spar(M), cost(M,C) }.\n"
    "seen(B) :- mass_band(B).\n"
    "#show spar/1.\n"
)

loose = RuleDiscipline(tied, facts=["mass_band"], produces="decisions")
try:
    loose.solve(mass_band="light")
except AmbiguousProgramError as error:
    for index, model in enumerate(error.models, 1):
        print(f"model {index}: {sorted(model)}")

model 1: ['spar(cfrp)']
model 2: ['spar(aluminium)']


`validate()` catches the same thing **statically**, by reading the program rather than solving
it. Note the trap it is looking for: `#minimize { 1@0,M : spar(M) }` *looks* like a tie-break and
separates nothing, because the weight is the same constant for every candidate.

In [8]:
from smartmdao import Pipeline, validate

loose_pipeline = Pipeline()
loose_pipeline.add(RuleDiscipline(tied, facts=["mass_band"], produces="decisions"))

@loose_pipeline.step(outputs=["report"])
def summarise(decisions: frozenset) -> str:
    return str(sorted(decisions))

for finding in validate(loose_pipeline, inputs=["mass_band"]):
    print(f"[{finding.code}] {finding.message}")

## Facts are symbols, not measurements

ASP has no floating point. Injecting `880.0` would mean choosing a threshold invisibly — exactly the
hypothesis [11 — Discretisation](11-discretisation.ipynb) exists to make visible. **The refusal is
the feature.**

In [9]:
from smartmdao import RuleProgramError

try:
    rules.solve(mass_band=880.0, certification="part23")
except RuleProgramError as error:
    print(error)

Fact 'mass_band' is the float 880.0. ASP has no floats, so injecting one means choosing a threshold - declare a Bands for 'mass_band' and pass the band instead. See docs/design/003.


## The whole bridge: numbers → band → facts → numbers

In [10]:
from smartmdao import Bands, Discretisation, analyze

wing = Pipeline(
    discretisation=Discretisation(
        mass_band=Bands("mass_kg", edges=[800], names=["light", "heavy"]),
    ),
)

@wing.step(outputs=["mass_kg"])
def size_airframe(span_m: float) -> float:
    return 120.0 * span_m

wing.add(RuleDiscipline(program, facts=["mass_band", "certification"], produces="decisions"))

@wing.step(outputs=["rib_pitch_m"])
def space_ribs(decisions: frozenset, span_m: float) -> float:
    return span_m / (8 if "spar(cfrp)" in decisions else 5)

print("order:", analyze(wing, inputs=["span_m", "certification"]).execution_order)
print()
for span in (5.0, 8.0):
    out = wing.run(span_m=span, certification="part23")
    print(f"span {span:4.1f} m -> {out['mass_kg']:6.1f} kg -> {out['mass_band']:6} "
          f"-> {sorted(out['decisions'])} -> pitch {out['rib_pitch_m']:.2f} m")

order: ('size_airframe', 'discretise_mass_band', 'rules_spar', 'space_ribs')

span  5.0 m ->  600.0 kg -> light  -> ['spar(aluminium)'] -> pitch 1.00 m
span  8.0 m ->  960.0 kg -> heavy  -> ['spar(cfrp)'] -> pitch 1.00 m


## What it costs

Answers are **memoised on their facts** — exact, not approximate, because the program is fixed and
clingo is deterministic. `RuleCost` reports what has actually been spent.

In [11]:
metered = RuleDiscipline(program, facts=["mass_band", "certification"],
                         produces="decisions", budget_seconds=5.0)

for _ in range(3):
    metered.solve(mass_band="light", certification="part23")
metered.solve(mass_band="heavy", certification="part23")

print(metered.cost)
print(f"projected for 30 sweeps: {metered.cost.projected_seconds(30):.4f}s")

4 call(s), 2 from cache, 2 ground+solve in 0.002s (0.001s grounding)
projected for 30 sweeps: 0.0293s


**`budget_seconds` bounds searching, not grounding.** Verified against clingo: a solve handle
cancels promptly, while an interrupt during grounding is ignored and grounding runs to completion.
Grounding is the worst-case-exponential half, so this is not protection against a grounding
blow-up — and a timeout is **not** `INFEASIBLE`, because a timeout proves nothing.

In [12]:
from smartmdao import RuleBudgetExceeded

hard = workspace / "pigeonhole.lp"
hard.write_text(
    "#const n=13.\n"
    "pigeon(1..n+1). hole(1..n).\n"
    "1 { in(P,H) : hole(H) } 1 :- pigeon(P).\n"
    ":- in(P1,H), in(P2,H), P1 < P2.\n"
    "seen(B) :- mass_band(B).\n"
    "#show in/2.\n"
)

slow = RuleDiscipline(hard, facts=["mass_band"], produces="decisions", budget_seconds=0.5)
try:
    slow.solve(mass_band="light")
except RuleBudgetExceeded as error:
    print(str(error).split(".")[0] + ".")
    print()
    print("Note: NOT INFEASIBLE. Nothing was proved about whether an answer exists.")

'/tmp/tmprdtftzwy/pigeonhole.lp' passed its 0.5s budget; cancelling the solve.


Solving '/tmp/tmprdtftzwy/pigeonhole.

Note: NOT INFEASIBLE. Nothing was proved about whether an answer exists.


## Where to put the decision

Inside a feedback loop the rules are applied **once per sweep, on values that have not settled** —
so the architecture is chosen from an artifact of the iteration path rather than from a result.
`validate()` reports it, as a warning rather than an error.

In [13]:
from smartmdao import HybridSolver

loop_rules = workspace / "loop.lp"
loop_rules.write_text(
    "spar(aluminium) :- mass_band(light).\n"
    "spar(cfrp)      :- mass_band(heavy).\n"
    "#show spar/1.\n"
)

inside = Pipeline(
    solver=HybridSolver(),
    discretisation=Discretisation(
        mass_band=Bands("total_mass_kg", edges=[500], names=["light", "heavy"]),
    ),
)
inside.add(RuleDiscipline(loop_rules, facts=["mass_band"], produces="decisions",
                          name="choose_spar"))

@inside.step(outputs=["structure_mass_kg"])
def size_structure(decisions: frozenset) -> float:
    return 200.0 if "spar(aluminium)" in decisions else 260.0

@inside.step(outputs=["total_mass_kg"])
def sum_masses(structure_mass_kg: float, payload_kg: float) -> float:
    return structure_mass_kg + payload_kg

for finding in validate(inside, inputs=["payload_kg"]):
    if finding.code.endswith("-in-cycle"):
        print(f"[{finding.code}]")
        print(f"   {finding.message}")
        print()

[rules-in-cycle]
   'choose_spar' applies the rules in '/tmp/tmprdtftzwy/loop.lp' from inside the loop choose_spar -> discretise_mass_band -> size_structure -> sum_masses, so it runs once per sweep and chooses from values that have not settled yet. A discrete choice inside a loop can leave it oscillating, or give it several stable answers that each report success. Putting the decision outside the cycle and iterating it explicitly avoids all of that - see docs/design/002.

[discretisation-in-cycle]
   'mass_band' is derived from a threshold inside the loop choose_spar -> discretise_mass_band -> size_structure -> sum_masses. A value crossing an edge mid-solve changes the band, which changes the result, which can move the value back - so this loop may settle in more than one place, each converged and self-consistent, chosen by the initial guess alone. Nothing reports which one you got.



The recommended shape is *decide → evaluate completely → revise*: keep the rules on the
**linear part**, run the pipeline with the decision fixed, read the new facts off the converged
result, and loop in ordinary Python.

What decides the topology is **one word** — whether a fact is named after the loop's own variable.

In [14]:
print('facts=["mass_band"]        -> cycles:',
      [c.steps for c in analyze(inside, inputs=["payload_kg"]).cycles])

outside_rules = workspace / "outside.lp"
outside_rules.write_text(
    "spar(aluminium) :- prior_mass_band(light).\n"
    "spar(cfrp)      :- prior_mass_band(heavy).\n"
    "#show spar/1.\n"
)

outside = Pipeline(
    solver=HybridSolver(),
    discretisation=Discretisation(
        mass_band=Bands("total_mass_kg", edges=[500], names=["light", "heavy"]),
    ),
)
outside.add(RuleDiscipline(outside_rules, facts=["prior_mass_band"], produces="decisions",
                           name="choose_spar"))
outside.add(size_structure, outputs=["structure_mass_kg"])
outside.add(sum_masses, outputs=["total_mass_kg"])

print('facts=["prior_mass_band"] -> cycles:',
      analyze(outside, inputs=["payload_kg", "prior_mass_band"]).cycles or "none")

facts=["mass_band"]        -> cycles: [('choose_spar', 'discretise_mass_band', 'size_structure', 'sum_masses')]
facts=["prior_mass_band"] -> cycles: none


Because the decision set is finite and the rules are deterministic, **a repeated decision set
is a cycle** — so memoising what you have seen gives termination, not just a retry cap.

---

**Next:** [13 — Execution and comparison](13-execution-and-comparison.ipynb).